## Chains

In [1]:
# import getpass
# import os
# os.environ["COHERE_API_KEY"] = getpass.getpass()
from langchain_core.prompts import ChatPromptTemplate
from langchain_cohere import ChatCohere
API_KEY = "o5S3pTQeNTMWAogDpDylcUSVhuYpcnwe1XOrnDdv"

In [2]:
chat_model = ChatCohere(cohere_api_key=API_KEY)

message_prompt = ChatPromptTemplate.from_messages([
    ("system" , "You are a math teacher."),
    ("human", "{first_number} plus {second_number} equals?")
])

In [3]:
chain = message_prompt | chat_model

In [4]:
chain_response = chain.invoke({"first_number":"Two thousand", "second_number":"nine hundred ninty nine"})
print(chain_response.content)

Two thousand plus nine hundred ninety-nine equals **2,999**.


## Parallel chains

In [5]:
## write a poem about a topic
message_prompt_poem = ChatPromptTemplate.from_messages([
    ("system", " تو یک شاعر ایرانی هستی "),
    ("human", "یک شعر دو بیتی درباره‌ی {topic} بگو")
    ])

chain_poem = message_prompt_poem | chat_model

%time
chain_response_poem = chain_poem.invoke({"topic": "پاییز"})
print(f'{chain_response_poem.content}')


CPU times: user 6 μs, sys: 2 μs, total: 8 μs
Wall time: 12.9 μs
برگ‌ها می‌رقصند در باد پاییز
خورشید می‌تابد، می‌شود دل تنگیز
رنگین شده باغ، چون نقشی زیبا
پاییز آمد و دل‌ها شد اسیر


In [6]:
## write a story about a topic
message_prompt_story = ChatPromptTemplate.from_messages([
    ("system", " تو یک نویسنده‌ی ایرانی هستی "),
    ("human", "یک داستان خیلی خیلی خیلی کوتاه درباره‌ی {topic}")
])

chain_story = message_prompt_story | chat_model
 
%time
chain_response_story = chain_story.invoke({"topic":"پاییز"})
print(f'{chain_response_story.content}')

CPU times: user 3 μs, sys: 1 μs, total: 4 μs
Wall time: 6.91 μs
**پاییز**  

برگ‌ها روی زمین می‌رقصیدند، انگار دارند آخرین ترانه‌شان را زمزمه کنند. نسیم خنک پاییزی از میان درختان می‌گذشت و بوی خاک تازه‌باران‌خورده را با خود می‌برد. دخترک کوچکی با چتر قرمزش در میان برگ‌های زرد و قرمز قدم برمی‌داشت. هر بار که پایش را برمی‌داشت، برگ‌ها زیر پای او می‌خندیدند. پاییز، فصل وداع بود، اما برای دخترک، شروع یک بازی جدید.


In [7]:
from langchain_core.runnables import RunnableParallel

combined = RunnableParallel(poem = chain_poem, story = chain_story)

%time
response = combined.invoke({'topic':'پاییز'})

CPU times: user 3 μs, sys: 0 ns, total: 3 μs
Wall time: 7.39 μs


In [8]:
print(response['poem'].content)

برگ‌ها می‌رقصند در باد پاییز
خاک می‌گیرد رنگی تازه و نو
عطر هیزم و باران، جان می‌بخشد
دل به طبیعت می‌سپارد، آرام و رو


## Writing functions in chains

In [30]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

message_prompt = ChatPromptTemplate.from_messages([
    ("system", " تو یک معلم ریاضی هستی "),
    ("human", "عدد {first_number} به علاوه‌ی {second_number} چند می‌شود؟")
])

model_chain = message_prompt | chat_model | StrOutputParser()

In [10]:
first_number, second_number = "۱۲", "شصت و دو"

chain_response = model_chain.invoke({"first_number": first_number, "second_number": second_number})
print(chain_response)

برای محاسبه‌ی عدد ۱۲ به علاوه‌ی شصت و دو، مراحل زیر را انجام می‌دهیم:

1. **تبدیل شصت و دو به عدد**:  
   شصت و دو به عدد برابر است با **۶۲**.

2. **جمع اعداد**:  
   $$
   ۱۲ + ۶۲ = ۷۴
   $$

**پاسخ نهایی**:  
عدد ۱۲ به علاوه‌ی شصت و دو برابر است با **۷۴**.  

$$
\boxed{74}
$$


In [11]:
import re

# Define a function to count the number of numbers in a text
def count_numbers(text):

    print(text)

    # Find all numbers in the text (including Persian and Arabic numbers)
    pattern = r'[0-9۰-۹]+'
    numbers = re.findall(pattern, text)

    # Return the number of numbers found in str format
    return str(len(numbers))

In [12]:
from langchain_core.runnables import RunnableLambda

count_runnable = RunnableLambda(count_numbers)

chain_counter = model_chain | count_runnable

In [13]:
chain_counter_response = chain_counter.invoke({"first_number": first_number, "second_number": second_number})

print(chain_counter_response)

عدد ۱۲ به علاوه‌ی شصت و دو برابر است با:

۱۲ + ۶۲ = ۷۴

پاسخ: **۷۴**
5


## RunnablePassThrough

In [31]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

final_output = RunnableParallel({
    "model_answer": RunnablePassthrough(),  
    "count_numbers_answer": count_runnable 
})

combined_chain = model_chain | final_output

In [32]:
response = combined_chain.invoke({"first_number": first_number, "second_number": second_number})
response

عدد ۱۲ به علاوه‌ی شصت و دو (۶۲) برابر است با:

۱۲ + ۶۲ = ۷۴

پاسخ: ۷۴


{'model_answer': 'عدد ۱۲ به علاوه\u200cی شصت و دو (۶۲) برابر است با:\n\n۱۲ + ۶۲ = ۷۴\n\nپاسخ: ۷۴',
 'count_numbers_answer': '6'}

In [37]:
model_chain = message_prompt | chat_model
response = model_chain.invoke({'first_number':'۱۲', 'second_number':'شصت ودو'})

## Debugging a chain

In [41]:
from operator import itemgetter

model = ChatCohere(cohere_api_key=API_KEY)

person_prompt = ChatPromptTemplate.from_template("What is the city {person} is from?")

sub_chain = person_prompt | model | StrOutputParser()

country_prompt = ChatPromptTemplate.from_template("what country is the city {city} in? Respond in {language}")

final_chain = {"city":sub_chain, "language":itemgetter("language")} | country_prompt | model | StrOutputParser()

In [45]:
response = final_chain.invoke({"person":"Khayyam", "language":"persian"})

In [46]:
print(response)

شهر نیشابور، زادگاه عمر خیام، ریاضیدان، ستارهشناس و شاعر نامدار ایرانی، در کشور ایران واقع شده است. نیشابور در شمال شرقی ایران قرار دارد و در دوران طلایی اسلام، یکی از مراکز مهم فرهنگی و فکری بود. این شهر نقش قابل توجهی در زندگی و آثار خیام داشته است. او به دلیل دستاوردهایش در ریاضیات، به ویژه در طبقهبندی و حل معادلات درجه سه، و همچنین به خاطر رباعیاتش که در سراسر جهان ترجمه و تحسین شدهاند، شناخته شده است.

**پاسخ:** ایران


In [ ]:
! pip install grandalf

In [49]:
final_chain.get_graph().print_ascii()

           +------------------------------+      
           | Parallel<city,language>Input |      
           +------------------------------+      
                  ***             ***            
                **                   ***         
              **                        **       
+--------------------+                    **     
| ChatPromptTemplate |                     *     
+--------------------+                     *     
           *                               *     
           *                               *     
           *                               *     
    +------------+                         *     
    | ChatCohere |                         *     
    +------------+                         *     
           *                               *     
           *                               *     
           *                               *     
  +-----------------+                 +--------+ 
  | StrOutputParser |                 | Lambda | 


#### Callbacks

In [51]:
from langchain.callbacks.tracers import ConsoleCallbackHandler

response = final_chain.invoke({"person": "Ferdowsi", "language": "Persian"}, config={'callbacks': [ConsoleCallbackHandler()]})
print(response)

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "person": "Ferdowsi",
  "language": "Persian"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<city,language>] Entering Chain run with input:
{
  "person": "Ferdowsi",
  "language": "Persian"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<city,language> > chain:RunnableSequence] Entering Chain run with input:
{
  "person": "Ferdowsi",
  "language": "Persian"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<city,language> > chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "person": "Ferdowsi",
  "language": "Persian"
}
[chain/end] [chain:RunnableSequence > chain:RunnableParallel<city,language> > chain:RunnableSequence > prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > chain:RunnableParallel<city,language> > chain:RunnableSequence > llm:ChatCohere] Entering LLM run wi

In [52]:
from langchain.globals import set_debug
set_debug(True)

In [53]:
response = final_chain.invoke({"person":"Khayyam", "language":"English"})
response

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "person": "Khayyam",
  "language": "English"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<city,language>] Entering Chain run with input:
{
  "person": "Khayyam",
  "language": "English"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<city,language> > chain:RunnableSequence] Entering Chain run with input:
{
  "person": "Khayyam",
  "language": "English"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<city,language> > chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:
{
  "person": "Khayyam",
  "language": "English"
}
[chain/end] [chain:RunnableSequence > chain:RunnableParallel<city,language> > chain:RunnableSequence > prompt:ChatPromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > chain:RunnableParallel<city,language> > chain:RunnableSequence > llm:ChatCohere] Entering LLM run with i

"Omar Khayyam was from **Iran**. Specifically, he was born in **Nishapur**, a city in northeastern Iran. Nishapur was a significant cultural and intellectual hub during the Islamic Golden Age, contributing greatly to the fields of science, literature, and art. Khayyam's legacy as a renowned Persian mathematician, astronomer, and poet is deeply intertwined with the historical importance of Nishapur in Iranian culture and history."